In [7]:
pip install torch torchvision torchaudio

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Definimos la arquitectura del "Cerebro" (Red Neuronal)
class CerebroIA(nn.Module):
    def __init__(self):
        super(CerebroIA, self).__init__()
        # Entrada: 3 sensores (Distancia frente, izquierda, derecha)
        self.capa_entrada = nn.Linear(3, 8) 
        # Capa oculta para procesar los datos
        self.capa_oculta = nn.Linear(8, 8)
        # Salida: 3 acciones (Izquierda, Derecha, Adelante)
        self.capa_salida = nn.Linear(8, 3)
        
    def forward(self, x):
        # Usamos la función de activación ReLU (la más común)
        x = torch.relu(self.capa_entrada(x))
        x = torch.relu(self.capa_oculta(x))
        x = self.capa_salida(x)
        return x

# 2. Inicializamos la Red
mi_red = CerebroIA()

# 3. Simulamos datos de sensores (Input)
# Digamos que el muro está cerca a la derecha: [Dist_Frente, Dist_Izq, Dist_Der]
datos_sensores = torch.tensor([0.8, 0.9, 0.2]) 

# 4. Le pedimos a la IA que tome una decisión
with torch.no_grad(): # No entrenamos, solo pedimos una predicción
    prediccion = mi_red(datos_sensores)
    accion = torch.argmax(prediccion).item()

# 5. Resultado lógico
acciones_texto = ["IZQUIERDA", "DERECHA", "ADELANTE"]
print(f"La IA decide ir hacia: {acciones_texto[accion]}")

La IA decide ir hacia: DERECHA


In [13]:
%pip install pygame neat-python

  Using cached pygame-2.6.1.tar.gz (14.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [112 lines of output]
      Skipping Cython compilation
      
      
      WARNING, No "Setup" File Exists, Running "buildconfig/config.py"
      Using WINDOWS configuration...
      
      Making dir :prebuilt_downloads:
      Downloading... https://www.libsdl.org/release/SDL2-devel-2.28.4-VC.zip 25ef9d201ce3fd5f976c37dddedac36bd173975c
      Unzipping :prebuilt_downloads\SDL2-devel-2.28.4-VC.zip:
      Downloading... https://www.libsdl.org/projects/SDL_image/release/SDL2_image-devel-2.0.5-VC.zip 137f86474691f4e12e76e07d58d5920c8d844d5b
      Unzipping :prebuilt_downloads\SDL2_image-devel-2.0.5-VC.zip:
      Downloading... https://github.com/libsdl-org/SDL_ttf/releases/download/release-2.20.1/SDL2_ttf-devel-2.20.1-VC.zip 371606aceba450384428fd2852f73d2f6290b136
      Unzipping :prebuilt_downloads\SDL2_ttf-devel-2.20.1-VC.zip:
      Downloading... https://g

In [2]:
%pip install neat-python

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install pygame-ce

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   --------------------------- ------------ 7.3/10.6 MB 46.5 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 42.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pygame
import neat
import os
import math

# Configuración de pantalla
WIDTH, HEIGHT = 600, 400
GOAL_POS = (550, 200)
OBSTACLE = pygame.Rect(250, 100, 50, 200)
START_POS = (50, 200)

class Ball:
    def __init__(self):
        self.x, self.y = START_POS
        self.radius = 10
        self.vel = 5
        self.dead = False

    def draw(self, win):
        pygame.draw.circle(win, (0, 100, 255), (int(self.x), int(self.y)), self.radius)

    def move(self, action):
        # action[0] controla arriba/abajo, action[1] izquierda/derecha
        if action[0] > 0.5: self.y -= self.vel
        if action[0] < -0.5: self.y += self.vel
        if action[1] > 0.5: self.x += self.vel
        if action[1] < -0.5: self.x -= self.vel

        # Bordes
        if self.x < 0 or self.x > WIDTH or self.y < 0 or self.y > HEIGHT:
            self.dead = True
        # Obstáculo
        if OBSTACLE.collidepoint(self.x, self.y):
            self.dead = True

def eval_genomes(genomes, config):
    nets = []
    ge = []
    balls = []

    for _, g in genomes:
        net = neat.nn.FeedForwardNetwork.create(g, config)
        nets.append(net)
        balls.append(Ball())
        g.fitness = 0
        ge.append(g)

    win = pygame.display.set_mode((WIDTH, HEIGHT))
    clock = pygame.time.Clock()
    
    run = True
    while run and len(balls) > 0:
        clock.tick(60)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run = False
                pygame.quit()

        for i, ball in enumerate(balls):
            # Inputs: Distancia X al objetivo, Distancia Y al objetivo, Distancia al obstáculo
            dist_x = GOAL_POS[0] - ball.x
            dist_y = GOAL_POS[1] - ball.y
            dist_obs = math.sqrt((ball.x - OBSTACLE.centerx)**2 + (ball.y - OBSTACLE.centery)**2)
            
            output = nets[i].activate((dist_x, dist_y, dist_obs))
            ball.move(output)

            # Recompensa por estar vivo y acercarse
            dist_to_goal = math.sqrt((ball.x - GOAL_POS[0])**2 + (ball.y - GOAL_POS[1])**2)
            ge[i].fitness = 1000 / (dist_to_goal + 1) # Cuanto más cerca, más fitness

            if ball.dead:
                ge[i].fitness -= 10 # Penalización por morir
                balls.pop(i)
                nets.pop(i)
                ge.pop(i)
            
            if dist_to_goal < 15: # Llegó a la meta
                ge[i].fitness += 100
                run = False

        # Dibujar
        win.fill((255, 255, 255))
        pygame.draw.rect(win, (255, 0, 0), OBSTACLE) # Pared
        pygame.draw.circle(win, (0, 255, 0), GOAL_POS, 15) # Meta
        for ball in balls:
            ball.draw(win)
        pygame.display.update()

def run(config_path):
    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         config_path)

    p = neat.Population(config)
    p.add_reporter(neat.StdOutReporter(True))
    p.run(eval_genomes, 50)

    if __name__ == "__main__":
        local_dir = os.path.dirname(__file__)
        config_path = os.path.join(local_dir, "config-feedforward.txt")
        run(config_path)

